In [1]:
!pip install wilds --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.5 MB/s eta 0:00:00


In [2]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from wilds import get_dataset
from wilds.common.data_loaders import get_train_loader, get_eval_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

/usr/local/lib/python3.12/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


cuda


In [3]:
root_dir = '/kaggle/input/datasets/rayyanshuda/waterbirds-wilds-v1/data'
dataset = get_dataset(dataset='waterbirds', download=True, root_dir=root_dir)
print(f"Total examples: {len(dataset)}")  # should be 11788

Total examples: 11788


In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),   # tuple, not an int for exact square resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = dataset.get_subset('train', transform=transform)
val_data = dataset.get_subset('val', transform=transform)
test_data = dataset.get_subset('test', transform=transform)

print(len(train_data), len(val_data), len(test_data))  # should be 4795 1199 5794

4795 1199 5794


In [5]:
BATCH_SIZE = 128

train_loader = get_train_loader('standard', train_data, batch_size=BATCH_SIZE)
val_loader = get_eval_loader('standard', val_data, batch_size=BATCH_SIZE)
test_loader = get_eval_loader('standard', test_data, batch_size=BATCH_SIZE)

In [6]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 234MB/s]


In [7]:
grouper = dataset._eval_grouper
n_groups = grouper.n_groups
group_names = ['landbird/land', 'landbird/water', 'waterbird/land', 'waterbird/water']

train_groups = grouper.metadata_to_group(train_data.metadata_array)
train_group_counts = torch.zeros(n_groups, dtype=torch.float)
train_group_counts.scatter_add_(0, train_groups, torch.ones_like(train_groups, dtype=torch.float))
print("Train group counts:", train_group_counts.tolist())  # should be [3498, 184, 56, 1057]

def evaluate(model, loader):
    model.eval()
    all_preds, all_y, all_metadata = [], [], []
    with torch.no_grad():
        for x, y, metadata in loader:
            x = x.to(device)
            logits = model(x)
            all_preds.append(logits.argmax(dim=1).cpu())
            all_y.append(y)
            all_metadata.append(metadata)
    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_y)
    metadata = torch.cat(all_metadata)

    g = grouper.metadata_to_group(metadata)
    correct = (y_pred == y_true).float()

    group_n = torch.zeros(n_groups, dtype=torch.float)
    group_n.scatter_add_(0, g, torch.ones_like(g, dtype=torch.float))
    group_correct = torch.zeros(n_groups, dtype=torch.float)
    group_correct.scatter_add_(0, g, correct)
    group_acc = group_correct / group_n.clamp(min=1)

    worst_group_acc = group_acc[group_n > 0].min().item()
    adj_acc_avg = (group_acc * train_group_counts).sum().item() / train_group_counts.sum().item()

    lines = [f"  {name}: {acc:.3f} (n={int(n)})"
             for name, acc, n in zip(group_names, group_acc.tolist(), group_n.tolist())]
    results_str = (f"Adjusted average acc: {adj_acc_avg:.3f}\n"
                    f"Worst-group acc: {worst_group_acc:.3f}\n" + "\n".join(lines))

    return adj_acc_avg, worst_group_acc, results_str

Train group counts: [3498.0, 184.0, 56.0, 1057.0]


In [8]:
sanity_subset = Subset(train_data, list(range(256)))
sanity_loader = DataLoader(sanity_subset, batch_size=32, shuffle=True)

sanity_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
sanity_model.fc = nn.Linear(sanity_model.fc.in_features, 2)
sanity_model = sanity_model.to(device)
sanity_optimizer = optim.SGD(sanity_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

for epoch in range(10): # 10 epochs
    sanity_model.train()
    total_loss = 0.0
    for x, y, metadata in sanity_loader:
        x, y = x.to(device), y.to(device)
        sanity_optimizer.zero_grad()
        loss = criterion(sanity_model(x), y)
        loss.backward()
        sanity_optimizer.step()
        total_loss += loss.item() * x.size(0)
    print(f"Sanity epoch {epoch+1}: loss = {total_loss / len(sanity_subset):.4f}")

Sanity epoch 1: loss = 0.6743
Sanity epoch 2: loss = 0.4401
Sanity epoch 3: loss = 0.2565
Sanity epoch 4: loss = 0.1410
Sanity epoch 5: loss = 0.0896
Sanity epoch 6: loss = 0.0673
Sanity epoch 7: loss = 0.0461
Sanity epoch 8: loss = 0.0362
Sanity epoch 9: loss = 0.0240
Sanity epoch 10: loss = 0.0295


In [9]:
NUM_EPOCHS = 300

best_wg_acc, best_avg_acc = -1, -1
best_wg_state, best_avg_state = None, None

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y, metadata in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_data)
    val_avg_acc, val_wg_acc, _ = evaluate(model, val_loader)

    if val_wg_acc > best_wg_acc:
        best_wg_acc = val_wg_acc
        best_wg_state = copy.deepcopy(model.state_dict())
    if val_avg_acc > best_avg_acc:
        best_avg_acc = val_avg_acc
        best_avg_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_avg_acc={val_avg_acc:.4f} | val_wg_acc={val_wg_acc:.4f}")

torch.save(best_wg_state, '/kaggle/working/erm_best_worst_group.pt')
torch.save(best_avg_state, '/kaggle/working/erm_best_avg.pt')

Epoch 1/300 | train_loss=0.3305 | val_avg_acc=0.9433 | val_wg_acc=0.1805
Epoch 2/300 | train_loss=0.1286 | val_avg_acc=0.9638 | val_wg_acc=0.3308
Epoch 3/300 | train_loss=0.0888 | val_avg_acc=0.9699 | val_wg_acc=0.4812
Epoch 4/300 | train_loss=0.0624 | val_avg_acc=0.9698 | val_wg_acc=0.4887
Epoch 5/300 | train_loss=0.0449 | val_avg_acc=0.9700 | val_wg_acc=0.4887
Epoch 6/300 | train_loss=0.0305 | val_avg_acc=0.9716 | val_wg_acc=0.5263
Epoch 7/300 | train_loss=0.0214 | val_avg_acc=0.9734 | val_wg_acc=0.4962
Epoch 8/300 | train_loss=0.0162 | val_avg_acc=0.9741 | val_wg_acc=0.5414
Epoch 9/300 | train_loss=0.0142 | val_avg_acc=0.9742 | val_wg_acc=0.5489
Epoch 10/300 | train_loss=0.0104 | val_avg_acc=0.9742 | val_wg_acc=0.5489
Epoch 11/300 | train_loss=0.0079 | val_avg_acc=0.9731 | val_wg_acc=0.5564
Epoch 12/300 | train_loss=0.0065 | val_avg_acc=0.9739 | val_wg_acc=0.5564
Epoch 13/300 | train_loss=0.0060 | val_avg_acc=0.9718 | val_wg_acc=0.5714
Epoch 14/300 | train_loss=0.0049 | val_avg_acc=

In [10]:
model.load_state_dict(best_wg_state)
avg1, wg1, str1 = evaluate(model, test_loader)
print("Best val worst-group acc")
print(str1)
print(f"Test avg acc: {avg1:.4f}, Test worst-group acc: {wg1:.4f}")

model.load_state_dict(best_avg_state)
avg2, wg2, str2 = evaluate(model, test_loader)
print("\nBest val average acc")
print(str2)
print(f"Test avg acc: {avg2:.4f}, Test worst-group acc: {wg2:.4f}")

Best val worst-group acc
Adjusted average acc: 0.972
Worst-group acc: 0.685
  landbird/land: 0.994 (n=2255)
  landbird/water: 0.720 (n=2255)
  waterbird/land: 0.685 (n=642)
  waterbird/water: 0.960 (n=642)
Test avg acc: 0.9725, Test worst-group acc: 0.6854

Best val average acc
Adjusted average acc: 0.973
Worst-group acc: 0.601
  landbird/land: 0.997 (n=2255)
  landbird/water: 0.789 (n=2255)
  waterbird/land: 0.601 (n=642)
  waterbird/water: 0.947 (n=642)
Test avg acc: 0.9733, Test worst-group acc: 0.6012
